In [1]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification
import re
from typing import List, Dict, Any

/home/harsh/projects/neomed/Neomed/backend/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_ner_pipeline():
    model_name = "blaze999/Medical-NER"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForTokenClassification.from_pretrained(model_name)
    
    ner_pipeline = pipeline(
        "ner",
        model=model,
        tokenizer=tokenizer,
        aggregation_strategy="simple",
        device=0
    )
    return ner_pipeline

In [3]:
def extract_entity_context(text: str, start: int, end: int, window: int = 40) -> str:
    left = max(0, start - window)
    right = min(len(text), end + window)
    return text[left:right]

In [12]:
ALLOWED_LABELS = {
    "DISEASE_DISORDER",
    "SIGN_SYMPTOM", 
    "MEDICATION",
    "THERAPEUTIC_PROCEDURE",
    "DIAGNOSTIC_PROCEDURE",
    "BIOLOGICAL_STRUCTURE",
    "DOSAGE",
    "LAB_VALUE",
    "AGE",
    "SEX",
    "DURATION"
}

In [13]:
def context_similarity(ctx1: str, ctx2: str) -> float:
    words1 = set(ctx1.lower().split())
    words2 = set(ctx2.lower().split())
    
    if not words1 or not words2:
        return 0.0
    
    intersection = len(words1 & words2)
    union = len(words1 | words2)
    
    return intersection / union if union > 0 else 0.0

In [14]:

def extract_medical_entities(text: str, ner_pipeline, context_window: int = 5, 
                            confidence_threshold: float = 0.6) -> Dict[str, Any]:
    """Extract and filter medical entities with proper context."""
    raw_entities = ner_pipeline(text)
    
    # Split into words while preserving positions
    words = text.split()
    
    # Build a mapping of character positions to word indices
    char_to_word_idx = {}
    char_pos = 0
    for word_idx, word in enumerate(words):
        for i in range(len(word)):
            char_to_word_idx[char_pos + i] = word_idx
        char_pos += len(word) + 1  # +1 for space
    
    entities = []
    
    for ent in raw_entities:
        # Filter by label and confidence
        if ent["entity_group"] not in ALLOWED_LABELS:
            continue
        if ent["score"] < confidence_threshold:
            continue
            
        start_char = ent["start"]
        end_char = ent["end"]
        
        # Find word indices using the mapping
        start_word_idx = char_to_word_idx.get(start_char, 0)
        end_word_idx = char_to_word_idx.get(end_char - 1, len(words) - 1)
        
        # Extract context (N words before and after)
        context_start = max(0, start_word_idx - context_window)
        context_end = min(len(words), end_word_idx + context_window + 1)
        context_words = words[context_start:context_end]
        
        entities.append({
            "text": ent["word"].strip(),
            "label": ent["entity_group"],
            "confidence": float(ent["score"]),
            "offsets": [(start_char, end_char)],
            "word_position": (start_word_idx, end_word_idx),
            "context": " ".join(context_words)
        })
    
    return {"text": text, "entities": entities}

In [15]:
def process_date_chunks(date_chunks: List[Dict[str, str]], ner_pipeline, 
                       context_window: int = 5, confidence_threshold: float = 0.5) -> List[Dict[str, Any]]:
    results = []
    
    for chunk in date_chunks:
        date = chunk.get("date", "N/A")
        text = chunk.get("text", "")
        
        if not text.strip():
            continue
        
        print(f"Processing date: {date}")
        
        extraction_result = extract_medical_entities(
            text, ner_pipeline, context_window, confidence_threshold
        )
        
        chunk_result = {
            "date": date,
            "entities": extraction_result["entities"]
        }
        
        results.append(chunk_result)
    
    return results


sample_input = [
    {
        "date": "2573-05-30",
        "text": clinical_note
    },
    {
        "date": "N/A",
        "text": "Patient presents with fever and cough for 3 days. Started on amoxicillin 500mg."
    }
]

results = process_date_chunks(sample_input, ner_pipeline)

import json
print("\n" + "="*60)
print("CLEAN JSON OUTPUT FOR LLM")
print("="*60)
print(json.dumps(results, indent=2))

Processing date: 2573-05-30
Processing date: N/A

CLEAN JSON OUTPUT FOR LLM
[
  {
    "date": "2573-05-30",
    "entities": [
      {
        "text": "Abdominal",
        "label": "BIOLOGICAL_STRUCTURE",
        "confidence": 0.8990191221237183,
        "offsets": [
          [
            238,
            248
          ]
        ],
        "word_position": [
          27,
          28
        ],
        "context": "Abdominal pain Major Surgical or Invasive Procedure: PICC line [**6-25**] ERCP w/"
      },
      {
        "text": "pain",
        "label": "SIGN_SYMPTOM",
        "confidence": 0.9385825991630554,
        "offsets": [
          [
            248,
            253
          ]
        ],
        "word_position": [
          28,
          28
        ],
        "context": "pain Major Surgical or Invasive Procedure: PICC line [**6-25**] ERCP w/"
      },
      {
        "text": "PICC line",
        "label": "THERAPEUTIC_PROCEDURE",
        "confidence": 0.9060595631599426,
    

In [16]:
clinical_note = """Admission Date:  [**2573-5-30**]              Discharge Date:   [**2573-7-1**]

Date of Birth:  [**2498-8-19**]             Sex:   F

Service: SURGERY

Allergies:
Hydrochlorothiazide

Attending:[**First Name3 (LF) 1893**]
Chief Complaint:
Abdominal pain

Major Surgical or Invasive Procedure:
PICC line [**6-25**]
ERCP w/ sphincterotomy [**5-31**]


History of Present Illness:
74y female with type 2 dm and a recent stroke affecting her
speech, who presents with 2 days of abdominal pain. Imaging shows no evidence of metastasis. She is not receiving any chemo.

Past Medical History:
1. Colon cancer dx'd in [**2554**], tx'd with hemicolectomy, XRT,
chemo. Last colonoscopy showed: Last CEA was in the 8 range
(down from 9)
2. Type II Diabetes Mellitus
3. Hypertension

Social History:
Married, former tobacco use. No alcohol or drug use.

Family History:
Mother with stroke at age 82. no early deaths.
2 daughters- healthy


Brief Hospital Course:
Ms. [**Known patient lastname 2004**] was admitted on [**2573-5-30**]. Ultrasound at the time of
admission demonstrated pancreatic duct dilitation and an
edematous gallbladder. She was admitted to the ICU.
Discharge Medications:
1. Miconazole Nitrate 2 % Powder Sig: One (1) Appl Topical  BID
(2 times a day) as needed.
2. Heparin Sodium (Porcine) 5,000 unit/mL Solution Sig: One (1)
Injection TID (3 times a day).
3. Acetaminophen 160 mg/5 mL Elixir Sig: One (1)  PO Q4-6H
(every 4 to 6 hours) as needed.

Discharge Diagnosis:
Type 2 DM
Pancreatitis
HTN
h/o aspiration respiratory distress


Discharge Instructions:
Patient may shower. Please call your surgeon or return to the
emergency room if [**Doctor First Name **] experience fever >101.5, nausea, vomiting,
abdominal pain, shortness of breath, abdominal pain or any
significant  change in your medical condition. A

Completed by: [**First Name11 (Name Pattern1) 2010**] [**Last Name (NamePattern1) 2011**] MD [**MD Number 2012**] [**2573-7-1**] @ 1404
Signed electronically by: DR. [**First Name8 (NamePattern2) **] [**Last Name (NamePattern1) **]
 on: FRI [**2573-7-2**] 8:03 AM
(End of Report)  """

In [6]:
ner_pipeline = load_ner_pipeline()

result = extract_medical_entities(clinical_note, ner_pipeline)

print("ENTITIES:")
for ent in result["entities"]:
    print(f"{ent['text']} | {ent['label']}")

Device set to use cuda:0
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


ENTITIES:
Abdominal | BIOLOGICAL_STRUCTURE
pain | SIGN_SYMPTOM
PICC line | THERAPEUTIC_PROCEDURE
ERCP | THERAPEUTIC_PROCEDURE
sphincterotomy | THERAPEUTIC_PROCEDURE
74y | AGE
female | SEX
2 days | DURATION
abdominal | BIOLOGICAL_STRUCTURE
pain | SIGN_SYMPTOM
Imaging | DIAGNOSTIC_PROCEDURE
chemo | MEDICATION
tx'd | THERAPEUTIC_PROCEDURE
hemicolectomy | THERAPEUTIC_PROCEDURE
XRT | THERAPEUTIC_PROCEDURE
colonoscopy | DIAGNOSTIC_PROCEDURE
CEA | DIAGNOSTIC_PROCEDURE
8 range | LAB_VALUE
Type II | DISEASE_DISORDER
Diabetes Mellitus | DISEASE_DISORDER
Ultrasound | DIAGNOSTIC_PROCEDURE
pancreatic duct | BIOLOGICAL_STRUCTURE
dilitation | SIGN_SYMPTOM
edematous | SIGN_SYMPTOM
gallbladder | BIOLOGICAL_STRUCTURE
Miconazole Nitrate | MEDICATION
2 % Powder | DOSAGE
2 times a day | DOSAGE
Heparin Sodium (Porcine) | MEDICATION
5,000 unit/mL Solution | DOSAGE
3 times a day | DOSAGE
Acetaminophen | MEDICATION
160 mg/5 mL | DOSAGE
Q4-6H | DOSAGE
every 4 to 6 hours | DOSAGE
2 | DISEASE_DISORDER
DM Pancreat

In [17]:
from collections import Counter

def analyze_entity_labels(entities: list) -> dict:
    """Discover all unique entity labels and their frequencies."""
    
    labels = [ent["label"] for ent in entities]
    label_counts = Counter(labels)
    
    print("=" * 60)
    print("ENTITY LABEL ANALYSIS")
    print("=" * 60)
    print(f"\nTotal entities found: {len(labels)}")
    print(f"Unique labels: {len(label_counts)}\n")
    
    print("Label frequencies:")
    print("-" * 60)
    for label, count in sorted(label_counts.items(), key=lambda x: x[1], reverse=True):
        percentage = (count / len(labels)) * 100
        print(f"{label:30} | Count: {count:3} | {percentage:5.1f}%")
    
    return {
        "unique_labels": list(label_counts.keys()),
        "label_counts": dict(label_counts),
        "total_entities": len(labels)
    }

# Run analysis
label_analysis = analyze_entity_labels(result["entities"])

# Show sample entities for each label
print("\n" + "=" * 60)
print("SAMPLE ENTITIES BY LABEL")
print("=" * 60)

entities_by_label = {}
for ent in result["entities"]:
    label = ent["label"]
    if label not in entities_by_label:
        entities_by_label[label] = []
    entry = ent["text"] + " : " + ent["context"]
    entities_by_label[label].append(entry)

for label in sorted(entities_by_label.keys()):
    samples = list(set(entities_by_label[label]))[:5]  # Show up to 5 unique examples
    print(f"\n{label}:")
    for sample in samples:
        print(f"  - {sample}")

ENTITY LABEL ANALYSIS

Total entities found: 48
Unique labels: 11

Label frequencies:
------------------------------------------------------------
SIGN_SYMPTOM                   | Count:  10 |  20.8%
DOSAGE                         | Count:   7 |  14.6%
BIOLOGICAL_STRUCTURE           | Count:   6 |  12.5%
THERAPEUTIC_PROCEDURE          | Count:   6 |  12.5%
DISEASE_DISORDER               | Count:   6 |  12.5%
DIAGNOSTIC_PROCEDURE           | Count:   4 |   8.3%
MEDICATION                     | Count:   4 |   8.3%
LAB_VALUE                      | Count:   2 |   4.2%
AGE                            | Count:   1 |   2.1%
SEX                            | Count:   1 |   2.1%
DURATION                       | Count:   1 |   2.1%

SAMPLE ENTITIES BY LABEL

AGE:
  - 74y : type 2 dm and a recent stroke affecting her speech, who presents

BIOLOGICAL_STRUCTURE:
  - gallbladder : was admitted to the ICU. Discharge Medications: 1. Miconazole Nitrate 2 %
  - abdominal : or any significant change in you